# Robust Pareto design for Cu–Ni–Si–Cr alloys
This notebook keeps all user-editable settings in one configuration cell. It retrains the locked XGBoost models, generates a large Sobol composition space, reconstructs the engineered features, removes out-of-domain candidates, and exports the HV–EC Pareto front and three representative alloys. Q3 is reported as an independent balance indicator and is not double-counted in the Pareto ranking.

In [ ]:
# ============================================================================
# USER CONFIGURATION TRUNK — normally edit only this cell
# ============================================================================
from pathlib import Path
import json, subprocess
import pandas as pd
from IPython.display import display, Image

WORK_DIR = Path('/Users/zixuanzhao/Desktop/Corpus–Topic–Document/ML2/消融实验/Pareto_Design')
SCRIPT = WORK_DIR / 'pareto_design.py'
CONFIG_PATH = WORK_DIR / 'pareto_config_runtime.json'
OUTPUT_DIR = WORK_DIR / 'results_formal'
PYTHON = '/opt/anaconda3/bin/python'

# Formal run: 2^17 = 131,072 initial Sobol candidates; use 2^18 for deeper search.
N_CANDIDATES = 131072
BOOTSTRAP_MODELS = 20       # 20–30 recommended for the paper
CONFIDENCE_Z = 1.0          # robust objective = mean prediction - z × bootstrap SD

# Composition search domain (wt.%). Cu is automatically calculated by balance.
# These bounds reproduce the existing Cu-combination experiment and stay close
# to the observed Cu–Ni–Si–Cr domain. Change only with metallurgical justification.
COMPOSITION_RANGES = {
    'Al': [0.30, 0.50],
    'Cr': [0.10, 0.30],
    'Mg': [0.01, 0.16],
    'Ni': [4.00, 6.00],
    'Si': [1.00, 1.40],
    'Zr': [0.00, 0.00],
}

# Composition-only Pareto experiment: processing is fixed for every candidate.
# Replace Aging_Temp/Aging_Time with the actual peak-aging condition before the
# final paper run. Solution_Time is exported but not used by the locked models.
FIXED_PROCESSING = {
    'Processing_Route': 2,
    'Solution_Temp': 980.0,
    'Solution_Time': 4.0,
    'CR_Reduction': 50.0,
    'Aging_Temp': 450.0,
    'Aging_Time': 4.0,
}

# Optional experimental alloy. Leave None until the measured composition is fixed.
EXPERIMENTAL_ALLOY = None

CONFIG = {
    'feature_file': '/Users/zixuanzhao/Desktop/Corpus–Topic–Document/FE/Feature/Feature3.xlsx',
    'raw_file': '/Users/zixuanzhao/Desktop/Corpus–Topic–Document/FE/Feature1.xlsx',
    'model_file': '/Users/zixuanzhao/Desktop/Corpus–Topic–Document/ML2/XGB.xlsx',
    'output_dir': str(OUTPUT_DIR),
    'random_seed': 42, 'test_size': 0.20,
    'n_candidates': N_CANDIDATES, 'bootstrap_models': BOOTSTRAP_MODELS,
    'confidence_z': CONFIDENCE_Z, 'ad_quantile': 0.95, 'ad_neighbors': 5,
    'fit_full_for_design': True,
    'composition_ranges_wt_pct': COMPOSITION_RANGES,
    'composition_rounding_wt_pct': {'Al':0.01,'Cr':0.01,'Mg':0.01,'Ni':0.02,'Si':0.01,'Zr':0.01},
    'constraints': {'cu_min_wt_pct': 92.0, 'ni_si_ratio': [3.5, 6.0]},
    'fixed_processing': FIXED_PROCESSING,
    'experimental_alloy': EXPERIMENTAL_ALLOY,
}
CONFIG_PATH.write_text(json.dumps(CONFIG, ensure_ascii=False, indent=2), encoding='utf-8')
print('Runtime configuration:', CONFIG_PATH)
print('Output directory:', OUTPUT_DIR)

In [ ]:
# Run the complete audited pipeline.
run = subprocess.run([PYTHON, str(SCRIPT), '--config', str(CONFIG_PATH)], text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError('Pareto pipeline failed; inspect the traceback above.')

In [ ]:
# Review the exact three experimental-design records and scientific audits.
three = pd.read_csv(OUTPUT_DIR / 'pareto_three_points.csv')
front = pd.read_csv(OUTPUT_DIR / 'pareto_front.csv')
audit = pd.read_csv(OUTPUT_DIR / 'model_reproduction_audit.csv')
display(three)
display(audit)
print(f'Robust Pareto-front candidates: {len(front)}')
display(Image(filename=str(OUTPUT_DIR / 'pareto_front.png')))